In [1]:
# STEP 1: Load dataset and provide overview
# Description: This script loads the EPC dataset and prepares it for roof geometry calculations.
# It will compute attic height, roof dimensions, and roof area based on given assumptions.

import pandas as pd
import numpy as np

# File paths
input_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_flat_update.csv"
output_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_roof_area_update.csv"

# Load dataset
df = pd.read_csv(input_path)

# Preview data
print(df.head())
print(df.columns)

   BUILDING_REFERENCE_NUMBER  OSG_REFERENCE_NUMBER              ADDRESS1  \
0                 1001829067           122009933.0  19 CULCREUCH AVENUE    
1                 1001951858           122010025.0             GLENVIEW    
2                 1001829071           122009868.0  22 CULCREUCH AVENUE    
3                 1234761001           122009968.0    1 MENZIES TERRACE    
4                 1001991633           122009892.0       50 MAIN STREET    

  ADDRESS2  ADDRESS3 POSTCODE   LATITUDE  LONGITUDE INSPECTION_DATE  \
0  FINTRY   GLASGOW   G63 0YB  56.055692  -4.223337         9/29/25   
1  FINTRY   GLASGOW   G63 0XJ  56.052824  -4.220640         9/19/25   
2  FINTRY   GLASGOW   G63 0YB  56.055503  -4.223691         7/29/25   
3  FINTRY   GLASGOW   G63 0YJ  56.058427  -4.224838         7/22/25   
4  FINTRY   GLASGOW   G63 0XF  56.054738  -4.228307         7/17/25   

         TYPE_OF_ASSESSMENT  ...   ROOF_AREA  USABLE_ROOF_LENGTH  \
0  RdSAP, existing dwelling  ...   29.698485    

In [2]:
# STEP 2: Create ATTIC_FLOOR_HEIGHT
# Description: Calculates attic floor height as NUMBER_OF_FLOORS * FLOOR_HEIGHT

df["ATTIC_FLOOR_HEIGHT"] = df["NUMBER_OF_FLOORS"] * df["FLOOR_HEIGHT"]

print(df[["NUMBER_OF_FLOORS", "FLOOR_HEIGHT", "ATTIC_FLOOR_HEIGHT"]].head())

   NUMBER_OF_FLOORS  FLOOR_HEIGHT  ATTIC_FLOOR_HEIGHT
0                 2           2.4                 4.8
1                 2           2.4                 4.8
2                 2           2.4                 4.8
3                 2           2.4                 4.8
4                 1           2.4                 2.4


In [3]:
# STEP 3: Create ROOF_LENGTH
# Description: Roof length is equal to WALL_LENGTH

df["ROOF_LENGTH"] = df["WALL_LENGTH"]

print(df[["WALL_LENGTH", "ROOF_LENGTH"]].head())

   WALL_LENGTH  ROOF_LENGTH
0     4.582576     4.582576
1     9.721111     9.721111
2     5.830952     5.830952
3     9.759611     9.759611
4    12.489996    12.489996


In [4]:
# STEP 4: Calculate ROOF_WIDTH
# Description:
# Roof width is the hypotenuse of a 45-45-90 triangle.
# One leg = (ROOF_RIDGE_HEIGHT - ATTIC_FLOOR_HEIGHT)
# Other leg = WALL_WIDTH / 2
# Hypotenuse = sqrt(leg^2 + leg^2)

# Calculate vertical leg
df["RIDGE_TO_ATTIC_HEIGHT"] = df["ROOF_RIDGE_HEIGHT"] - df["ATTIC_FLOOR_HEIGHT"]

# Calculate half width
df["HALF_WALL_WIDTH"] = df["WALL_WIDTH"] / 2

# Compute hypotenuse (roof slope length)
df["ROOF_WIDTH"] = np.sqrt(df["RIDGE_TO_ATTIC_HEIGHT"]**2 + df["HALF_WALL_WIDTH"]**2)

print(df[["RIDGE_TO_ATTIC_HEIGHT", "HALF_WALL_WIDTH", "ROOF_WIDTH"]].head())

   RIDGE_TO_ATTIC_HEIGHT  HALF_WALL_WIDTH  ROOF_WIDTH
0               2.291288         2.291288    3.240370
1               3.240370         3.240370    4.582576
2               2.915476         2.915476    4.123106
3               3.253204         3.253204    4.600725
4               6.563332         4.163332    7.772429


In [5]:
# STEP 5: Calculate ROOF_AREA and HALF_ROOF_AREA
# Description:
# ROOF_AREA = ROOF_LENGTH * ROOF_WIDTH
# HALF_ROOF_AREA = ROOF_AREA / 2

df["HALF_ROOF_AREA"] = df["ROOF_LENGTH"] * df["ROOF_WIDTH"]
df["ROOF_AREA"] = df["HALF_ROOF_AREA"] * 2

print(df[["ROOF_LENGTH", "ROOF_WIDTH", "ROOF_AREA", "HALF_ROOF_AREA"]].head())

   ROOF_LENGTH  ROOF_WIDTH   ROOF_AREA  HALF_ROOF_AREA
0     4.582576    3.240370   29.698485       14.849242
1     9.721111    4.582576   89.095454       44.547727
2     5.830952    4.123106   48.083261       24.041631
3     9.759611    4.600725   89.802561       44.901281
4    12.489996    7.772429  194.155227       97.077613


In [6]:
# STEP 6: Save updated dataset
# Description: Writes the modified dataset with new columns to CSV

df.to_csv(output_path, index=False)

print(f"File saved to: {output_path}")

File saved to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_roof_area_update.csv
